In [1]:
import pandas as pd
import numpy as np

# optional: display settings for Jupyter
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 120)

In [2]:
# Seed data: UK figures for two years
seed_data = [
    {
        "Region": "UK",
        "Year": 2023,
        "Sales_GBP_million": 72.1,
        "Operating_Profit_GBP_million": 3.3,
        # Profit margin, calculated as operating profit / sales
        "Op_Profit_Margin_pct": 100 * 3.3 / 72.1
    },
    {
        "Region": "UK",
        "Year": 2024,
        "Sales_GBP_million": 81.4,
        "Operating_Profit_GBP_million": 8.0,
        "Op_Profit_Margin_pct": 100 * 8.0 / 81.4
    }
]

df_seed = pd.DataFrame(seed_data)
df_seed

,Region,Year,Sales_GBP_million,Operating_Profit_GBP_million,Op_Profit_Margin_pct
0,UK,2023,72.1,3.3,4.576976
1,UK,2024,81.4,8.0,9.828010


In [3]:
regions = ["UK", "USA", "APAC", "EMEA", "Middle East"]
years = [2022, 2023, 2024]  # example range

In [4]:
# Cross-join regions and years
df_template = pd.MultiIndex.from_product([regions, years], names=["Region", "Year"]).to_frame(index=False)

df_template.head()

,Region,Year
0,UK,2022
1,UK,2023
2,UK,2024
3,USA,2022
4,USA,2023


In [5]:
# Merge seed where available
df = df_template.merge(df_seed, on=["Region", "Year"], how="left")

df.head(10)

,Region,Year,Sales_GBP_million,Operating_Profit_GBP_million,Op_Profit_Margin_pct
0,UK,2022,NaN,NaN,NaN
1,UK,2023,72.1,3.3,4.576976
2,UK,2024,81.4,8.0,9.828010
3,USA,2022,NaN,NaN,NaN
4,USA,2023,NaN,NaN,NaN
5,USA,2024,NaN,NaN,NaN
6,APAC,2022,NaN,NaN,NaN
7,APAC,2023,NaN,NaN,NaN
8,APAC,2024,NaN,NaN,NaN
9,EMEA,2022,NaN,NaN,NaN


In [6]:
np.random.seed(42)

# Define base sales in GBP millions by region (illustrative)
base_sales = {
    "UK": 70,
    "USA": 200,
    "APAC": 150,
    "EMEA": 100,
    "Middle East": 90
}

# Growth estimates per year (illustrative)
year_growth = {
    2022: 1.00,
    2023: 1.10,
    2024: 1.18
}

# Typical profit margin range (illustrative) in %
profit_margin_min = 8
profit_margin_max = 20

# Fill missing values
for idx, row in df.iterrows():
    if pd.isna(row["Sales_GBP_million"]):
        region = row["Region"]
        year = row["Year"]
        base = base_sales.get(region, 50)
        growth = year_growth.get(year, 1.0)
        # simulate sales
        sales = base * growth * (0.9 + 0.2 * np.random.rand())  # random ±10%
        df.at[idx, "Sales_GBP_million"] = round(sales, 1)
    if pd.isna(row["Operating_Profit_GBP_million"]):
        sales = df.at[idx, "Sales_GBP_million"]
        # simulate profit margin
        margin = profit_margin_min + (profit_margin_max - profit_margin_min) * np.random.rand()
        profit = sales * margin / 100.0
        df.at[idx, "Operating_Profit_GBP_million"] = round(profit, 1)
        df.at[idx, "Op_Profit_Margin_pct"] = round(margin, 1)

df.head(10)

,Region,Year,Sales_GBP_million,Operating_Profit_GBP_million,Op_Profit_Margin_pct
0,UK,2022,68.2,13.2,19.400000
1,UK,2023,72.1,3.3,4.576976
2,UK,2024,81.4,8.0,9.828010
3,USA,2022,209.3,31.8,15.200000
4,USA,2023,204.9,20.2,9.900000
5,USA,2024,215.1,39.6,18.400000
6,APAC,2022,153.0,25.2,16.500000
7,APAC,2023,149.2,29.3,19.600000
8,APAC,2024,188.8,19.9,10.500000
9,EMEA,2022,93.6,9.5,10.200000


In [7]:
def synthetic_demographics(region):
    # Example templates per region; adjust as appropriate
    # Values represent averages or percentages
    if region == "UK":
        return {
            "Avg_Customer_Age": 45,
            "Pct_High_Income": 0.65,
            "Pct_Male": 0.70,
            "Pct_Female": 0.30,
            "Pct_Local_vs_Tourist": 0.4  # local proportion
        }
    elif region == "USA":
        return {
            "Avg_Customer_Age": 43,
            "Pct_High_Income": 0.60,
            "Pct_Male": 0.68,
            "Pct_Female": 0.32,
            "Pct_Local_vs_Tourist": 0.5
        }
    elif region == "APAC":
        return {
            "Avg_Customer_Age": 41,
            "Pct_High_Income": 0.55,
            "Pct_Male": 0.65,
            "Pct_Female": 0.35,
            "Pct_Local_vs_Tourist": 0.6
        }
    elif region == "EMEA":
        return {
            "Avg_Customer_Age": 44,
            "Pct_High_Income": 0.58,
            "Pct_Male": 0.66,
            "Pct_Female": 0.34,
            "Pct_Local_vs_Tourist": 0.5
        }
    elif region == "Middle East":
        return {
            "Avg_Customer_Age": 42,
            "Pct_High_Income": 0.62,
            "Pct_Male": 0.67,
            "Pct_Female": 0.33,
            "Pct_Local_vs_Tourist": 0.55
        }
    else:
        # defaults
        return {
            "Avg_Customer_Age": 43,
            "Pct_High_Income": 0.55,
            "Pct_Male": 0.65,
            "Pct_Female": 0.35,
            "Pct_Local_vs_Tourist": 0.5
        }

# Apply synthetic demographics
demo_cols = [
    "Avg_Customer_Age",
    "Pct_High_Income",
    "Pct_Male",
    "Pct_Female",
    "Pct_Local_vs_Tourist"
]

for idx, row in df.iterrows():
    region = row["Region"]
    demo = synthetic_demographics(region)
    for col, val in demo.items():
        df.at[idx, col] = val

df.head(10)

,Region,Year,Sales_GBP_million,Operating_Profit_GBP_million,Op_Profit_Margin_pct,Avg_Customer_Age,Pct_High_Income,Pct_Male,Pct_Female,Pct_Local_vs_Tourist
0,UK,2022,68.2,13.2,19.400000,45.0,0.65,0.70,0.30,0.4
1,UK,2023,72.1,3.3,4.576976,45.0,0.65,0.70,0.30,0.4
2,UK,2024,81.4,8.0,9.828010,45.0,0.65,0.70,0.30,0.4
3,USA,2022,209.3,31.8,15.200000,43.0,0.60,0.68,0.32,0.5
4,USA,2023,204.9,20.2,9.900000,43.0,0.60,0.68,0.32,0.5
5,USA,2024,215.1,39.6,18.400000,43.0,0.60,0.68,0.32,0.5
6,APAC,2022,153.0,25.2,16.500000,41.0,0.55,0.65,0.35,0.6
7,APAC,2023,149.2,29.3,19.600000,41.0,0.55,0.65,0.35,0.6
8,APAC,2024,188.8,19.9,10.500000,41.0,0.55,0.65,0.35,0.6
9,EMEA,2022,93.6,9.5,10.200000,44.0,0.58,0.66,0.34,0.5


In [8]:
# Reorder columns for readability
columns_order = [
    "Region", "Year", "Sales_GBP_million", "Operating_Profit_GBP_million", "Op_Profit_Margin_pct",
    "Avg_Customer_Age", "Pct_High_Income", "Pct_Male", "Pct_Female", "Pct_Local_vs_Tourist"
]

df_final = df[columns_order].copy()
df_final

,Region,Year,Sales_GBP_million,Operating_Profit_GBP_million,Op_Profit_Margin_pct,Avg_Customer_Age,Pct_High_Income,Pct_Male,Pct_Female,Pct_Local_vs_Tourist
0,UK,2022,68.2,13.2,19.400000,45.0,0.65,0.70,0.30,0.40
1,UK,2023,72.1,3.3,4.576976,45.0,0.65,0.70,0.30,0.40
2,UK,2024,81.4,8.0,9.828010,45.0,0.65,0.70,0.30,0.40
3,USA,2022,209.3,31.8,15.200000,43.0,0.60,0.68,0.32,0.50
4,USA,2023,204.9,20.2,9.900000,43.0,0.60,0.68,0.32,0.50
5,USA,2024,215.1,39.6,18.400000,43.0,0.60,0.68,0.32,0.50
6,APAC,2022,153.0,25.2,16.500000,41.0,0.55,0.65,0.35,0.60
7,APAC,2023,149.2,29.3,19.600000,41.0,0.55,0.65,0.35,0.60
8,APAC,2024,188.8,19.9,10.500000,41.0,0.55,0.65,0.35,0.60
9,EMEA,2022,93.6,9.5,10.200000,44.0,0.58,0.66,0.34,0.50


In [9]:
df_final.isnull().sum()

Region                          0
Year                            0
Sales_GBP_million               0
Operating_Profit_GBP_million    0
Op_Profit_Margin_pct            0
Avg_Customer_Age                0
Pct_High_Income                 0
Pct_Male                        0
Pct_Female                      0
Pct_Local_vs_Tourist            0
dtype: int64

In [10]:
# Round values to compare
df_final["Calc_Margin"] = (df_final["Operating_Profit_GBP_million"] / df_final["Sales_GBP_million"] * 100).round(1)

# Compare to stored margin
df_final[["Region","Year","Op_Profit_Margin_pct","Calc_Margin"]].head(10)

,Region,Year,Op_Profit_Margin_pct,Calc_Margin
0,UK,2022,19.400000,19.4
1,UK,2023,4.576976,4.6
2,UK,2024,9.828010,9.8
3,USA,2022,15.200000,15.2
4,USA,2023,9.900000,9.9
5,USA,2024,18.400000,18.4
6,APAC,2022,16.500000,16.5
7,APAC,2023,19.600000,19.6
8,APAC,2024,10.500000,10.5
9,EMEA,2022,10.200000,10.1


In [11]:
# Save to CSV
df_final.to_csv("ap_sales_demographics_dataset.csv", index=False)

# Optional: save to Excel
df_final.to_excel("ap_sales_demographics_dataset.xlsx", index=False)